# Palmer Penguins | Flipper Length and Body Mass

Gerardo Oburu Ndong · November 23, 2026  
CIS215 Capstone 12.6

## Initial data exploration

I’m using the simplified Palmer Penguins CSV. The [data notes](../data/README.md) explain where it came from and who was sampled.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

%matplotlib inline

# I am going to support a kernel started at the project root or in notebooks/.
project_root = Path("..") if Path("../data/README.md").is_file() else Path(".")
data_path = project_root / "data" / "raw" / "penguins.csv"
if not data_path.is_file():
    raise FileNotFoundError("Open the notebook from the project root or notebooks folder.")

penguins_raw = pd.read_csv(data_path)
penguins_raw.info()
display(penguins_raw.describe(include="all").round(2))
display(penguins_raw.isna().sum().rename("missing_rows").to_frame())
print(f"Exact duplicate rows: {penguins_raw.duplicated().sum()}")

The exploration output shows 344 rows and eight columns. Both bill columns, flipper length, and body mass each have two missing values. Recorded sex is missing in 11 rows. I found no exact duplicate rows. The simplified file has no bird ID, so that check only tells me whether entire rows match.

## Data cleaning and preparation

| Column | How I handled missing values |
|---|---|
| Species | No missing values. A row without a species could not go into a species comparison. |
| Island | No missing values. Island is used to check coverage. |
| Bill length | Two missing values. I left them blank because bill length does not enter my analysis. |
| Bill depth | Two missing values. I left them blank for the same reason. |
| Flipper length | Two missing values. I excluded those rows from the paired analysis because a missing flipper length cannot be plotted or correlated with mass. |
| Body mass | Two missing values, on the same rows as flipper length. I excluded those rows once and kept the original CSV intact. |
| Sex | Eleven missing values in the original file. I kept unknown sex in the pooled and species summaries when both measurements exist. I used known values for the sex comparison. |
| Year | No missing values. I kept the original year to describe the field seasons. |

I trimmed spaces in the group labels, standardized recorded sex to lowercase, and changed the display spelling of Adelie to Adélie. I checked for nonpositive dimensions and used kilograms on charts because the labels are easier to read.

In [ ]:
cleaned = penguins_raw.copy()
for column in ["species", "island", "sex"]:
    cleaned[column] = cleaned[column].str.strip().replace("", pd.NA)
cleaned["sex"] = cleaned["sex"].str.lower()
cleaned["species"] = cleaned["species"].replace({"Adelie": "Adélie"})

measurements = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
assert not cleaned[measurements].le(0).any().any(), "Check nonpositive measurements"
assert not cleaned[["species", "island", "year"]].isna().any().any()
assert not cleaned.duplicated().any(), "Review matching rows before proceeding"

paired = cleaned.dropna(subset=["flipper_length_mm", "body_mass_g"]).copy()
paired["body_mass_kg"] = paired["body_mass_g"] / 1000
print(f"Paired analysis: {len(paired)} of {len(cleaned)} rows")
print(f"Paired rows with unknown sex: {paired['sex'].isna().sum()}")